In [0]:
!PYTHONPATH=. PYTHONDONTWRITEBYTECODE=1 pytest /Workspace/Users/soumyamukherjee42@gmail.com/personal_development/tests/


In [0]:
from pyspark.sql import functions as F

##  Curated Tables 

In [0]:
def save_as_curated_table(df, table_name: str):
    """
    Saves a DataFrame as a managed Delta table with 'curated_' prefix.
    """
    full_table_name = f"curated_{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_table_name)
    print(f"✅ Created table: {full_table_name}")


## **Customers Tables**

In [0]:
# Read raw customer table
raw_customers = spark.sql(
    """
    select * from raw_customers
    """
)

# ---------------------------
# Curated Customer Master
# ---------------------------
curated_customer_master = (
    raw_customers.select(
        "customer_id",
        "customer_name",
        "email",
        "phone",
        "segment"
    )
    .dropDuplicates(["customer_id"])
)

curated_customer_master.write.format("delta").mode("overwrite").saveAsTable("curated_customer_master")
print("✅ curated_customer_master created")

# ---------------------------
# Curated Customer Address
# ---------------------------
curated_customer_address = (
    raw_customers.select(
        "customer_id",
        "address",
        "city",
        "state",
        "postal_code",
        "country",
        "region"
    )
    .dropDuplicates(["customer_id"])
)

curated_customer_address.write.format("delta").mode("overwrite").saveAsTable("curated_customer_address")
print("✅ curated_customer_address created")

## Products Tables

In [0]:
# Read raw products table
raw_products = spark.table("raw_products")

# ---------------------------
# Curated Products Table
# ---------------------------
curated_products = (
    raw_products.selectExpr(
        "product_id",
        "category",
        "sub_category",
        "product_name",
        "state",
        "price_per_product"
    )
    .dropDuplicates(["product_id"])
)

curated_products.write.format("delta").mode("overwrite").saveAsTable("curated_products")
print("✅ curated_products created")


## Orders Tables

In [0]:
from pyspark.sql import functions as F

# ---------------------------
# Read raw orders and curated products
# ---------------------------
raw_orders = spark.table("raw_orders")
curated_products = spark.table("curated_products")

# ---------------------------
# Curated Order Header
# ---------------------------
curated_order_header = (
    raw_orders.select(
        "order_id",
        "order_date",
        "ship_date",
        "ship_mode",
        "customer_id"
    ).dropDuplicates(["order_id"])
)

curated_order_header.write.format("delta").mode("overwrite").saveAsTable("curated_order_header")
print("✅ curated_order_header created")

# ---------------------------
# Curated Order Details
# ---------------------------
curated_order_details = (
    raw_orders.join(curated_products, on="product_id", how="left")
    .select(
        raw_orders["row_id"].cast("long").alias("row_id"),
        raw_orders["order_id"],
        raw_orders["product_id"],
        curated_products["category"],
        curated_products["sub_category"],
        curated_products["product_name"],
        raw_orders["quantity"].cast("long").alias("quantity"),
        raw_orders["price"].cast("double").alias("price"),
        raw_orders["discount"].cast("double").alias("discount"),
        F.round(raw_orders["profit"].cast("double"), 2).alias("profit"),
    )
)

curated_order_details.write.format("delta").mode("overwrite").saveAsTable("curated_order_details")
print("✅ curated_order_details created")
